# 📝 과제 01: 온라인 쇼핑 상담 모델 SFT 학습과 에폭별 체크포인트 저장

## 만들 결과

교안 01(SFT 지도 파인튜닝: SFTTrainer로 노무 상담 모델 학습하기)의 SFT를 지난 시간 과제의 온라인 쇼핑 상담(전자상거래법) 데이터에 같은 순서로 적용합니다. 에폭마다 저장한 체크포인트는 과제 02에서 판사로 채점해 쓸 모델을 고릅니다.

**만들 것**
- 같은 질문 여섯 개에 학습 전 모델(Base)과 학습한 모델들이 한 답을 나란히 둔 `check_table`
- 학습·검증 Dataset `train_dataset`, `val_dataset`
- 토큰 길이 테이블과 `max_length` 판단(가장 긴 예시 122토큰)
- 학습 설정 `sft_config`와, 학습 전에 손실 대상 토큰을 확인한 학습기 `trainer`
- 학습 곡선과 검증 손실 기록 `eval_log`
- lr 5e-6로 4에폭 학습하며 에폭마다 저장한 체크포인트 `output/commerce_epochs/checkpoint-<갱신 횟수>` 네 개(과제 02가 평가합니다)

## 준비: 데이터와 제공 코드

- **자료:** `data/과제/commerce_train.jsonl`(학습 데이터), `data/과제/commerce_val.jsonl`(검증 데이터), 교안 01과 같은 일반 질문 `data/본실습/general_questions.jsonl`(정답 답변 키는 `reference_answer`). 지난 시간 과제에서 원본(`origin_id`) 단위로 나눠 저장한 질문-답변이고, 행마다 `question`, `answer`, `messages`(system, user, assistant) 등이 있습니다. 범위 밖 질문 행의 `answer`는 거절 답 `REFUSAL_ANSWER_COMMERCE`입니다.
- **제공 코드:** `data_dir`, `output_dir`, `device`, `MODEL_NAME`, `tokenizer`, `SYSTEM_PROMPT_COMMERCE`, `REFUSAL_ANSWER_COMMERCE`, 교안 01과 같은 함수 `generate_reply(model, question, system_prompt=SYSTEM_PROMPT_COMMERCE)`, `load_jsonl(path)`, `to_prompt_completion(row)`, MLflow 준비(실험 `commerce-sft`)
- **교안 01과 다른 점:** 학습 데이터가 135행이라 1에폭이 34번 갱신으로 짧아, 첫 학습과 학습률 비교는 10번 갱신마다 검증 손실을 계산합니다. 예시도 짧아 `max_length`를 256으로 둡니다. 학습률과 에폭은 교안 01과 같습니다.
- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다. GPU 메모리 24GB 이상 RunPod 파드에서 실행하고, 7번 체크포인트 네 개가 디스크를 약 12GB 씁니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
# 지난 시간 과제 데이터의 시스템 프롬프트와 범위 밖 거절 답입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
REFUSAL_ANSWER_COMMERCE = "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_reply(model, question, system_prompt=SYSTEM_PROMPT_COMMERCE, max_new_tokens=256):
    """질문 하나를 학습 데이터와 같은 chat template으로 감싸 모델이 새로 만든 답만 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
        enable_thinking=False,       # 빈 생각 블록(<think></think>)을 붙여 추론 과정 없이 바로 답합니다
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한. 답이 이보다 길면 끝까지 쓰기 전에 멈춥니다
            do_sample=False,                       # 매번 확률이 가장 높은 토큰을 골라(그리디 생성) 같은 질문에 같은 답이 나옵니다
            eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
            pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
        )

    # 입력 프롬프트 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("commerce-sft")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

## 1. 학습 데이터와 학습 전 모델의 답

**배경**: 교안 01의 학습 전 결과와 같은 방법입니다. 같은 질문 여섯 개를 학습 전 모델에 묻고, 학습한 모델의 답은 6~7번에서 같은 테이블에 열로 더해 답이 어떻게 바뀌는지 봅니다. `base_model`은 5번에서 지우므로 답을 지금 모아 둡니다.

**요구사항**
- (1) `data/과제/commerce_train.jsonl`을 `load_jsonl`로 읽은 행 딕셔너리 리스트 **`train_rows`** 와, `data/과제/commerce_val.jsonl`을 읽은 **`val_rows`** 를 만듭니다.
- (2) 학습 전 모델을 `MODEL_NAME`에서 `dtype=torch.float32`로 불러 `device`에 올린 **`base_model`** 을 만듭니다.
- (3) 셀에 적어 둔 질문 묶음 `check_rows`(학습 행 둘, 검증 데이터 첫 행, 학습 데이터의 첫 거절 답 행, 일반 질문 둘)와 그 종류 `check_kinds`를 그대로 씁니다. `check_rows`의 `question`마다 `generate_reply`로 만든 답을 같은 순서의 리스트로 돌려주는 함수 **`answer_check_questions(model)`** 을 만듭니다. 종류가 `"일반 질문"`이면 `system_prompt=None`(시스템 프롬프트 없이), 나머지는 기본 시스템 프롬프트로 묻습니다.
- (4) 열이 `"종류"`(`check_kinds`), `"질문"`, `"정답 답변"`(행의 `answer`, 일반 질문 행은 `reference_answer`), `"Base 답변"`(`answer_check_questions(base_model)`)이고 이 순서인 테이블 **`check_table`** 을 만듭니다.

**확인 기준**
- `train_rows`, `val_rows`의 행 수가 자가채점 셀의 기대값과 같고, `base_model`의 가중치가 `float32`입니다.
- `check_table`은 여섯 행이고 열 이름과 순서가 위와 같으며, `"Base 답변"`은 빈 문자열이 아닌 문자열입니다.
- `answer_check_questions`는 일반 질문 행(`check_kinds`가 `"일반 질문"`)을 `system_prompt=None`으로 묻습니다. 자가채점 셀은 답의 내용을 보지 않으므로 이 조건은 함수 코드에서 직접 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 데이터 불러오기, 학습 전 모델 불러오기, 학습 전 결과 셀을 합친 모양입니다. 같은 질문을 뒤에서 여러 모델에 다시 물으므로, 질문을 묻는 일은 함수 하나로 둡니다.

세부구현:
1. 경로는 data_dir 아래 과제 폴더입니다.
2. 모델은 AutoModelForCausalLM의 from_pretrained에 MODEL_NAME과 float32 자료형을 넘긴 뒤 device로 옮깁니다.
3. 함수 안에서 zip으로 check_rows와 check_kinds를 함께 돌며, 종류에 따라 generate_reply에 system_prompt=None을 넘길지 정하고 반환값을 리스트에 더합니다.
4. 정답 답변은 행에 answer 키가 있으면 그 값, 없으면 reference_answer 값을 for 문으로 모읍니다. pd.DataFrame에는 열 이름을 키, 리스트를 값으로 한 사전을 넘깁니다.
```

</details>

In [ ]:
# [작성] (1) 학습·검증 파일을 읽고, (2) 학습 전 모델을 불러, (3) 질문 묶음에 답하는 함수를 만들고, (4) Base 답을 테이블로 만드세요.

# (1) 지난 시간 과제에서 원본 단위로 나눠 저장한 학습, 검증 파일을 그대로 읽습니다
train_rows = ...
val_rows = ...
print("학습 행:", len(train_rows), "| 검증 행:", len(val_rows))

# (2) 학습 전 모델(Base): 학습할 때와 같이 float32로 불러옵니다
base_model = ...

# (3) 학습 데이터에서 정답 답변이 거절 답인 첫 행(그대로 씁니다)
for row in train_rows:
    if row["answer"] == REFUSAL_ANSWER_COMMERCE:
        refusal_row = row
        break

# 쇼핑과 상관없는 상식 질문(교안 01과 같은 파일, 정답 답변은 reference_answer)
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 학습 전후에 똑같이 물을 질문 여섯 개와 그 종류(그대로 씁니다)
check_rows = [train_rows[0], train_rows[90], val_rows[0], refusal_row, general_rows[0], general_rows[12]]
check_kinds = ["학습 데이터", "학습 데이터", "검증 데이터", "범위 밖 질문", "일반 질문", "일반 질문"]


def answer_check_questions(model):
    """check_rows의 질문마다 model이 한 답을 같은 순서의 리스트로 돌려줍니다. 일반 질문은 시스템 프롬프트 없이 묻습니다."""
    answers = []
    ...
    return answers


# (4) 질문마다 정답 답변: 쇼핑 상담 행은 answer, 일반 질문은 reference_answer
reference_answers = ...

# 종류, 질문, 정답 답변, Base 답을 한 테이블에 둡니다. 학습한 모델의 답은 6번부터 이 테이블에 열로 더합니다
check_table = ...
display(check_table)

In [ ]:
# [자가채점] 행 수, 학습 전 모델의 자료형, 질문 묶음 테이블의 열과 행을 확인합니다
assert isinstance(train_rows, list) and isinstance(val_rows, list), "train_rows, val_rows는 load_jsonl로 읽은 행 딕셔너리의 리스트입니다. `...`를 채웠는지 확인하세요."
assert len(train_rows) == 135 and len(val_rows) == 34, f"학습 {len(train_rows)}행, 검증 {len(val_rows)}행입니다. data/과제의 commerce_train.jsonl, commerce_val.jsonl을 그대로 읽으세요(기대값: 학습 135행, 검증 34행)."
assert next(base_model.parameters()).dtype == torch.float32, "base_model은 MODEL_NAME에서 dtype=torch.float32로 불러옵니다."
assert callable(answer_check_questions), "answer_check_questions는 model 하나를 받아 답 리스트를 돌려주는 함수입니다."

assert isinstance(check_table, pd.DataFrame), "check_table은 pd.DataFrame으로 만든 테이블입니다."
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "Base 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. 열은 \"종류\", \"질문\", \"정답 답변\", \"Base 답변\" 순서입니다."
assert len(check_table) == 6 and len(check_rows) == 6, f"check_table이 {len(check_table)}행입니다. 셀에 적어 둔 check_rows 여섯 행을 그대로 쓰세요."
expected_questions = []
expected_answers = []
for row in check_rows:
    expected_questions.append(row["question"])
    if "answer" in row:
        expected_answers.append(row["answer"])
    else:
        expected_answers.append(row["reference_answer"])
assert list(check_table["종류"]) == check_kinds, "\"종류\" 열은 셀에 적어 둔 check_kinds를 순서 그대로 담습니다."
assert list(check_table["질문"]) == expected_questions and list(check_table["정답 답변"]) == expected_answers, "\"질문\", \"정답 답변\" 열은 check_rows의 question과 answer(일반 질문은 reference_answer)를 순서 그대로 담습니다."
for answer in check_table["Base 답변"]:
    assert isinstance(answer, str) and answer.strip() != "", "Base 답변이 비어 있습니다. answer_check_questions(base_model)이 질문마다 generate_reply의 반환값을 담는지 확인하세요."
print("✅ 통과!")

## 2. prompt·completion Dataset

**배경**: 교안 01의 prompt와 completion 소절과 같습니다. 마지막 assistant 메시지를 completion으로 떼어 주면 학습기는 답변 토큰만 손실로 계산합니다.

**요구사항**
- (1) 행마다 `to_prompt_completion`을 적용한 리스트를 `Dataset.from_list`에 넘겨 **`train_dataset`**(`train_rows`)과 **`val_dataset`**(`val_rows`)을 만듭니다.

**확인 기준**
- 두 Dataset의 열은 `prompt`, `completion`이고 예시 수가 행 수와 같습니다. 첫 예시의 `prompt` 역할(`role`)은 system, user이고 `completion`은 assistant 하나입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 prompt·completion 셀과 같습니다. 행마다 나눈 딕셔너리로 Dataset을 만듭니다.

세부구현:
1. to_prompt_completion을 적용한 딕셔너리 리스트를 Dataset.from_list에 넘깁니다. 검증 데이터도 같습니다.
```

</details>

In [ ]:
# [작성] (1) Dataset을 만드세요.
from datasets import Dataset

# (1) messages를 prompt(system, user)와 completion(assistant)으로 나눠 SFTTrainer가 받는 Dataset으로 만듭니다
train_dataset = ...
val_dataset = ...
display(train_dataset[0])

In [ ]:
# [자가채점] 두 Dataset의 열과 메시지 역할을 확인합니다
assert isinstance(train_dataset, Dataset) and isinstance(val_dataset, Dataset), "train_dataset, val_dataset은 Dataset.from_list로 만든 Dataset입니다. `...`를 채웠는지 확인하세요."
assert train_dataset.column_names == ["prompt", "completion"], f"train_dataset의 열이 {train_dataset.column_names}입니다. 행마다 to_prompt_completion을 적용한 딕셔너리 리스트로 만드세요."
assert val_dataset.column_names == ["prompt", "completion"], f"val_dataset의 열이 {val_dataset.column_names}입니다. val_rows에도 to_prompt_completion을 적용하세요."
assert len(train_dataset) == len(train_rows) and len(val_dataset) == len(val_rows), "학습, 검증 행마다 예시가 하나씩 있어야 합니다."

# 첫 학습 예시의 메시지 역할
prompt_roles = []
for message in train_dataset[0]["prompt"]:
    prompt_roles.append(message["role"])
completion_roles = []
for message in train_dataset[0]["completion"]:
    completion_roles.append(message["role"])
assert prompt_roles == ["system", "user"], f"prompt의 역할이 {prompt_roles}입니다. to_prompt_completion에 행(row) 전체를 넘기세요."
assert completion_roles == ["assistant"], f"completion의 역할이 {completion_roles}입니다. to_prompt_completion에 행(row) 전체를 넘기세요."
print("✅ 통과!")

## 3. 토큰 길이와 max_length

**배경**: `max_length`보다 긴 예시는 끝의 답과 대화 끝 토큰 `<|im_end|>`가 잘린 채 학습되므로, 학습 설정 전에 토큰 수를 셉니다(교안 01의 토큰 길이와 max_length). 이 데이터는 짧아 상한 후보를 256으로 둡니다.

**요구사항**
- (1) `train_rows + val_rows`의 행마다 `messages`를 `tokenizer.apply_chat_template`로 펼친 글(토큰화 끔)의 토큰 수와, `answer`만의 토큰 수를 셉니다. 두 수를 열 `"예시 토큰 수"`, `"답변 토큰 수"`로 담은 테이블 **`length_table`** 을 만듭니다.
- (2) 가장 긴 예시의 토큰 수를 **`longest_example`**, 가장 긴 답변의 토큰 수를 **`longest_answer`** 에 담습니다.
- (3) 학습 설정에 넣을 상한 256이 가장 긴 예시 이상인지(참·거짓)를 **`fits_max_length`** 에 담습니다.

**확인 기준**
- `length_table`은 학습·검증 행 수만큼이고 열이 위 순서입니다. `longest_example`(가장 긴 예시의 토큰 수)은 122, `longest_answer`(가장 긴 답변의 토큰 수)는 49로 자가채점 셀의 기대값과 같고, `fits_max_length`는 참입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01 학습 전 점검의 토큰 길이 셀과 같습니다. 행마다 펼친 글과 답의 토큰 수를 세고, 테이블 열의 최댓값을 상한과 비교합니다.

세부구현:
1. 펼친 글은 apply_chat_template에 messages를 넣고 토큰화를 끄면 문자열로 받습니다.
2. 토큰 수는 tokenizer(글)이 돌려준 input_ids의 길이입니다.
3. 최댓값은 length_table의 열마다 max로 구합니다.
```

</details>

In [ ]:
# [작성] (1) 토큰 수 테이블을 만들고, (2) 가장 긴 값을 찾은 뒤, (3) 상한 256과 비교하세요.

# (1) 학습·검증 예시의 전체 토큰 수(chat template으로 펼친 글)와 답변 토큰 수를 셉니다
full_lengths = []
answer_lengths = []
for row in train_rows + val_rows:
    ...
length_table = pd.DataFrame({"예시 토큰 수": full_lengths, "답변 토큰 수": answer_lengths})
display(length_table.describe().round(1))

# (2) 가장 긴 예시와 가장 긴 답변의 토큰 수
longest_example = ...
longest_answer = ...

# (3) 상한이 가장 긴 예시 이상이면 잘리는 예시가 없습니다
fits_max_length = ...
print("가장 긴 예시:", longest_example, "토큰 | 가장 긴 답변:", longest_answer, "토큰 | max_length 256 충분:", fits_max_length)

In [ ]:
# [자가채점] 토큰 수 테이블의 열과 행 수, 가장 긴 예시와 답변, 상한 판단을 확인합니다
assert isinstance(length_table, pd.DataFrame), "length_table은 pd.DataFrame으로 만든 테이블입니다. `...`를 채웠는지 확인하세요."
assert list(length_table.columns) == ["예시 토큰 수", "답변 토큰 수"], f"length_table의 열이 {list(length_table.columns)}입니다. 열 이름은 \"예시 토큰 수\", \"답변 토큰 수\"입니다."
assert len(length_table) == len(train_rows) + len(val_rows), "length_table은 train_rows + val_rows의 행마다 한 행입니다."
assert longest_example == length_table["예시 토큰 수"].max() and longest_answer == length_table["답변 토큰 수"].max(), "longest_example, longest_answer는 length_table 두 열의 가장 큰 값입니다."
assert longest_example == 122, f"가장 긴 예시가 {longest_example}토큰입니다(기대값 122). messages 전체를 apply_chat_template로 펼친 글(tokenize=False)을 토큰화했는지 확인하세요."
assert longest_answer == 49, f"가장 긴 답변이 {longest_answer}토큰입니다(기대값 49). 행의 answer 문자열만 토큰화하세요."
assert bool(fits_max_length) == (256 >= longest_example) and bool(fits_max_length), "fits_max_length는 256이 가장 긴 예시 이상인지 비교한 값입니다."
print("✅ 통과!")

## 4. 학습 설정과 학습 전 점검

**배경**: 교안 01의 학습 설정과 학습기, 답변 토큰만 손실 소절과 같습니다. 학습기를 만든 뒤 `train`을 부르기 전에 첫 예시의 `labels`를 확인합니다.

**요구사항**
- (1) `SFTConfig`에 아래 값을 넣어 **`sft_config`** 를 만듭니다.

```text
output_dir                   str(output_dir / "commerce_lr2e-5")
learning_rate                2e-5
num_train_epochs             1
per_device_train_batch_size  4
save_strategy                "no"
seed                         0
gradient_checkpointing       False
max_length                   256
eval_strategy                "steps"
eval_steps                   10
logging_steps                10
eval_on_start                True
report_to                    ["mlflow"]
run_name                     "lr_2e-5"
```

- (2) 한 에폭의 갱신 횟수를 정수 **`steps_per_epoch`** 에 담습니다. 학습 예시 수를 배치 크기로 나눠 올림(`math.ceil`)합니다.
- (3) 학습할 모델을 `MODEL_NAME`에서 `dtype=torch.float32`로 새로 불러 `device`에 올린 **`first_model`** 을 만들고, `SFTTrainer`에 `first_model`, `sft_config`, `train_dataset`, `eval_dataset=val_dataset`, `processing_class=tokenizer`를 넘긴 학습기 **`trainer`** 를 만듭니다. 학습은 5번에서 합니다.
- (4) 학습기가 토큰화한 첫 학습 예시 `trainer.train_dataset[0]`을 토큰마다 펼쳐, 글자(`tokenizer.decode`로 토큰 하나씩 바꾼 문자열), `labels`, 손실 대상 여부(`labels`가 -100이 아니면 `True`)를 열 `"토큰"`, `"labels"`, `"손실 대상"`으로 담은 테이블 **`token_table`** 을 만듭니다.
- (5) 손실 대상 토큰 수를 정수 **`n_loss_tokens`** 에, 손실 대상 토큰 가운데 마지막 두 토큰의 글자를 문자열 리스트 **`last_loss_tokens`** 에 담습니다.

**확인 기준**
- `sft_config`의 값이 위와 같고 `steps_per_epoch`가 자가채점 셀의 기대값과 같습니다.
- `first_model`의 가중치가 `float32`이고, `trainer`의 `model`은 `first_model`, `args`는 `sft_config`이며 아직 갱신하지 않았습니다(`trainer.state.global_step`이 0). `trainer.eval_dataset`은 검증 행 수만큼입니다.
- `token_table`의 맨 앞 토큰은 손실 대상이 아니고, `last_loss_tokens`는 `["<|im_end|>", "\n"]`입니다(대화 끝 토큰과 줄바꿈까지 손실 대상).

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 학습 설정, 학습기 만들기, 학습 예시 테이블 셀과 같은 순서이고, 한 에폭의 갱신 횟수는 교안 01 에폭별 체크포인트 셀과 같은 계산입니다. 학습기를 만드는 순간 토큰화와 labels가 만들어지므로 train을 부르지 않고 확인합니다.

세부구현:
1. SFTConfig는 위 값을 키워드 인자로 넣습니다. output_dir는 문자열로 넘깁니다.
2. 올림은 math 모듈의 ceil입니다. 배치 크기는 sft_config에서 읽습니다.
3. 학습기는 model, args, train_dataset, eval_dataset, processing_class 다섯 키워드로 만듭니다.
4. 손실 대상 열은 labels 열이 -100과 다른지 비교한 결과이고, 손실 대상 수는 이 열의 True 개수입니다.
5. 손실 대상인 행만 남긴 테이블에서 토큰 열의 마지막 두 값을 리스트로 바꿉니다.
```

</details>

In [ ]:
# [작성] (1) 학습 설정을 만들고, (2) 한 에폭의 갱신 횟수를 계산하세요.
import math

from trl import SFTConfig, SFTTrainer

# (1) 교안 01 sft_config와 같은 방법으로, 이 데이터에 맞는 값을 넣습니다
sft_config = SFTConfig(
    ...
)

# (2) 한 에폭의 갱신 횟수 = 올림(학습 예시 수 / 배치 크기). 마지막 배치는 남은 예시만으로 한 번 갱신합니다
steps_per_epoch = ...
print("한 에폭의 갱신 횟수:", steps_per_epoch, "| 전체 갱신 횟수:", steps_per_epoch * sft_config.num_train_epochs)

In [ ]:
# [자가채점] 학습 설정 값과 한 에폭의 갱신 횟수를 확인합니다
assert isinstance(sft_config, SFTConfig), "sft_config는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert sft_config.num_train_epochs == 1 and sft_config.max_steps == -1 and sft_config.per_device_train_batch_size == 4, "num_train_epochs=1, per_device_train_batch_size=4를 넣고 max_steps는 넣지 마세요(max_steps를 주면 에폭 수보다 앞섭니다)."
assert abs(sft_config.learning_rate - 2e-5) < 1e-12, f"learning_rate가 {sft_config.learning_rate}입니다. 2e-5를 넣으세요."
assert sft_config.max_length == 256, f"max_length가 {sft_config.max_length}입니다. 3번에서 확인한 상한 256을 넣으세요."
assert sft_config.eval_strategy == "steps" and sft_config.eval_steps == 10, "eval_strategy=\"steps\", eval_steps=10를 넣으세요."
assert sft_config.logging_steps == 10, "logging_steps는 10입니다(학습 손실도 검증 손실과 같은 간격)."
assert sft_config.eval_on_start is True, "eval_on_start=True를 넣어야 학습 전(step 0) 검증 손실이 남습니다."
assert sft_config.save_strategy == "no" and sft_config.seed == 0, "save_strategy=\"no\", seed=0을 넣으세요."
assert sft_config.gradient_checkpointing is False, "gradient_checkpointing=False를 넣으세요(trl 기본값은 켜짐)."
assert "mlflow" in sft_config.report_to and sft_config.run_name == "lr_2e-5", "report_to=[\"mlflow\"], run_name=\"lr_2e-5\"를 넣으세요."
assert sft_config.output_dir.endswith("commerce_lr2e-5"), f"output_dir가 {sft_config.output_dir}입니다. str(output_dir / \"commerce_lr2e-5\")를 넣으세요."
assert steps_per_epoch == 34, f"한 에폭의 갱신 횟수가 {steps_per_epoch}입니다(기대값 34). len(train_dataset)을 per_device_train_batch_size로 나눠 math.ceil로 올리세요."
print("✅ 통과!")

In [ ]:
# [작성] (3) 학습할 모델과 학습기를 만들고, (4) 첫 학습 예시를 테이블로 펼쳐 (5) 손실 대상을 확인하세요.

# (3) 학습할 모델을 학습 전 상태로 새로 불러옵니다. 학습이 끝나면 이 모델이 lr 2e-5 모델입니다
first_model = ...

# 학습기를 만들면 학습·검증 데이터셋을 토큰화하고 labels를 만듭니다(아직 학습하지 않습니다)
trainer = ...

# (4) 학습기가 토큰화한 첫 학습 예시를 토큰마다 펼칩니다(-100은 손실에서 빼는 표시)
example = trainer.train_dataset[0]
token_table = pd.DataFrame({
    "토큰": ...,
    "labels": ...,
})
token_table["손실 대상"] = ...

# (5) 손실 대상 토큰 수와, 손실 대상 가운데 마지막 두 토큰
n_loss_tokens = ...
last_loss_tokens = ...
print("전체 토큰:", len(token_table), "| 손실 대상 토큰:", n_loss_tokens, "| 마지막 두 손실 대상:", last_loss_tokens)
display(token_table.tail(4))

In [ ]:
# [자가채점] 학습 전 학습기, 토큰 테이블, 손실 대상 토큰 수와 마지막 두 토큰을 확인합니다
assert next(first_model.parameters()).dtype == torch.float32, "first_model은 MODEL_NAME에서 dtype=torch.float32로 불러옵니다. `...`를 채웠는지 확인하세요."
assert isinstance(trainer, SFTTrainer) and trainer.args is sft_config, "trainer는 sft_config를 args로 넘긴 SFTTrainer입니다."
assert trainer.model is first_model, "SFTTrainer의 model에 (3)에서 불러온 first_model을 넘기세요."
assert trainer.state.global_step == 0, "4번에서는 학습기를 만들기만 합니다. 학습(train)은 5번에서 하세요. 이미 학습했다면 4번 셀부터 다시 실행하세요."
assert len(trainer.eval_dataset) == len(val_dataset), "학습기에 eval_dataset=val_dataset을 넘기세요. 5번 검증 손실을 이 데이터로 계산합니다."

# 첫 학습 예시에서 손실 대상(labels가 -100이 아닌) 토큰 수
first_labels = trainer.train_dataset[0]["labels"]
expected_loss_tokens = 0
for label in first_labels:
    if label != -100:
        expected_loss_tokens += 1

assert list(token_table.columns) == ["토큰", "labels", "손실 대상"], f"token_table의 열이 {list(token_table.columns)}입니다. 열은 \"토큰\", \"labels\", \"손실 대상\" 순서입니다."
assert len(token_table) == len(trainer.train_dataset[0]["input_ids"]), "token_table은 첫 학습 예시의 input_ids 토큰마다 한 행입니다."
assert not token_table["손실 대상"].iloc[0], "맨 앞 토큰이 손실 대상입니다. 2번처럼 prompt, completion으로 나눈 train_dataset을 학습기에 넘겼는지 확인하세요."
assert n_loss_tokens == expected_loss_tokens, f"n_loss_tokens가 {n_loss_tokens}입니다. 손실 대상 열의 True 개수({expected_loss_tokens})를 담으세요."
assert 0 < n_loss_tokens < len(token_table), "손실 대상은 답변 토큰뿐이라 전체 토큰 수보다 적습니다."
assert last_loss_tokens == ["<|im_end|>", "\n"], f"last_loss_tokens가 {last_loss_tokens}입니다. 손실 대상인 행만 남긴 뒤 토큰 열의 마지막 두 값을 리스트로 담으세요."
print("✅ 통과!")

## 5. 학습과 학습 곡선

**배경**: 교안 01의 학습 곡선 소절과 같습니다. 학습 전 답은 1번 테이블에 모았으므로 먼저 `base_model`을 지워 GPU 메모리를 비웁니다.

**요구사항**
- (1) `base_model`을 `del`로 지우고 `gc.collect()`, `torch.cuda.empty_cache()`로 GPU 메모리를 비웁니다.
- (2) `trainer`로 학습하고 반환값을 **`train_output`** 에 담습니다.
- (3) `trainer.state.log_history`를 테이블로 바꾼 **`log_table`** 을 만듭니다. `"loss"`가 있는 행의 `step`, `loss` 두 열만 남긴 **`train_log`** 와, `"eval_loss"`가 있는 행의 `step`, `eval_loss` 두 열만 남긴 **`eval_log`** 를 만듭니다.
- (4) 가로축 `step`으로 학습 손실(색 `#2563eb`)과 검증 손실(색 `#dc2626`)을 한 그래프(`figsize=(7, 4)`)에 그리고 한글 제목, 축 이름, 범례를 붙입니다.

**확인 기준**
- `base_model` 이름이 남아 있지 않고, 1에폭 끝까지 돌아 `train_output.global_step`이 34입니다. `log_table`은 `trainer.state.log_history`의 기록 수만큼입니다.
- `eval_log`(열 `step`, `eval_loss`)의 `step`은 0, 10, 20, 30, 34(0은 학습 전), `train_log`(열 `step`, `loss`)의 `step`은 10, 20, 30입니다. 마지막 학습 손실이 첫 학습 손실보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 학습 셀과 학습 곡선 셀을 이어 붙인 모양입니다. 기록 리스트를 테이블로 바꾸고, 손실 열이 비어 있지 않은 행만 골라 그립니다.

세부구현:
1. del 뒤에 gc 모듈을 불러와 정리하고, 학습기의 train을 부릅니다.
2. dropna의 subset으로 한 열이 비어 있지 않은 행만 남긴 뒤 두 열만 고릅니다.
3. plt.plot을 두 번 부르고 title, xlabel, ylabel, legend를 붙입니다.
```

</details>

In [ ]:
# [작성] (1) base_model을 지우고, (2) 학습한 뒤, (3) 기록을 테이블로 나눠 (4) 학습 곡선을 그리세요.
import gc

# (1) 학습 전 답은 1번의 check_table에 모았으므로 base_model을 지우고 GPU 메모리를 비웁니다
...

# (2) 순전파 -> 손실 -> 역전파 -> 갱신을 1에폭 동안 반복하고, 정한 간격마다 검증 손실을 계산합니다
train_output = ...
display(train_output)

# (3) 학습기 기록: 한 줄에 학습 손실(loss)이나 검증 손실(eval_loss) 가운데 하나가 있고, 마지막 줄은 학습 전체 요약입니다
log_table = ...
train_log = ...
eval_log = ...
display(eval_log.round(4))

# (4) 학습 손실(파랑)과 검증 손실(빨강)을 갱신 횟수 축에 그립니다. 검증 손실은 학습 전(step 0)부터 있습니다
plt.figure(figsize=(7, 4))
...
plt.show()

In [ ]:
# [자가채점] 지운 모델, 끝까지 돈 학습, 두 손실 테이블의 열과 step, 학습 손실의 방향을 확인합니다
assert "base_model" not in globals(), "학습 전에 base_model을 del로 지워 GPU 메모리를 비우세요(학습 전 답은 check_table에 있습니다)."
assert train_output.global_step == 34, f"global_step이 {train_output.global_step}입니다. trainer.train()이 끝까지 돌았는지 확인하세요."
assert isinstance(log_table, pd.DataFrame) and len(log_table) == len(trainer.state.log_history), "log_table은 trainer.state.log_history 전체를 테이블로 바꾼 것입니다."
assert list(eval_log.columns) == ["step", "eval_loss"], f"eval_log의 열이 {list(eval_log.columns)}입니다. step, eval_loss 두 열만 남기세요."
assert list(eval_log["step"]) == [0, 10, 20, 30, 34], f"eval_log의 step이 {list(eval_log['step'])}입니다. log_table에서 eval_loss가 비어 있지 않은 행만 남기세요(eval_on_start=True면 step 0이 있습니다)."
assert list(train_log.columns) == ["step", "loss"], f"train_log의 열이 {list(train_log.columns)}입니다. step, loss 두 열만 남기세요."
assert list(train_log["step"]) == [10, 20, 30], f"train_log의 step이 {list(train_log['step'])}입니다. log_table에서 loss가 비어 있지 않은 행만 남기세요."
assert train_log["loss"].iloc[-1] < train_log["loss"].iloc[0], "마지막 학습 손실이 첫 학습 손실보다 작지 않습니다. 2번 train_dataset과 4번 learning_rate(2e-5)를 확인하고 4번 셀부터 다시 실행하세요."
print("✅ 통과!")

## 6. 학습률 비교

**배경**: 교안 01의 학습률 비교와 같습니다. lr 2e-5 모델의 답을 모은 뒤 저장하지 않고 지우고(GPU 한 장에 학습기 둘이 들어가지 않음), 학습률만 줄인 학습(lr 5e-6)을 같은 MLflow 실험에 남겨 두 학습률의 답을 나란히 봅니다. 두 run의 검증 손실 곡선은 교안 01의 두 학습의 검증 손실처럼 MLflow 화면에서 겹쳐 볼 수 있습니다(채점하지 않습니다).

**요구사항**
- (1) `answer_check_questions(first_model)`로 만든 답을 `check_table`의 새 열 `"lr 2e-5 답변"`에 담습니다.
- (2) lr 2e-5 모델은 저장하지 않습니다. `trainer`, `first_model`을 `del`로 지운 뒤 `gc.collect()`, `torch.cuda.empty_cache()`로 GPU 메모리를 비웁니다.
- (3) `sft_config`와 같은 값에서 `learning_rate=5e-6`, `output_dir=str(output_dir / "commerce_lr5e-6")`, `run_name="lr_5e-6"` 세 가지만 바꾼 **`small_lr_config`** 를 만듭니다. 나머지 `num_train_epochs`, `per_device_train_batch_size`, `save_strategy`, `seed`, `gradient_checkpointing`, `max_length`, `eval_strategy`, `eval_steps`, `logging_steps`, `eval_on_start`, `report_to`는 4번 값 그대로 넣습니다.
- (4) 학습 전 모델을 `MODEL_NAME`에서 `dtype=torch.float32`로 새로 불러 `device`에 올린 **`small_lr_model`** 을 만들고, `SFTTrainer`에 `small_lr_model`, `small_lr_config`, `train_dataset`, `eval_dataset=val_dataset`, `processing_class=tokenizer`를 넘긴 학습기 **`small_lr_trainer`** 를 만들어 학습한 반환값을 **`small_lr_output`** 에 담습니다.
- (5) `answer_check_questions(small_lr_model)`로 만든 답을 `check_table`의 새 열 `"lr 5e-6 답변"`에 담고 테이블을 보여 줍니다.
- (6) lr 5e-6 모델도 저장하지 않습니다. `small_lr_model`, `small_lr_trainer`를 지운 뒤 GPU 메모리를 비웁니다.

**확인 기준**
- `check_table`의 열은 1번의 열 뒤에 `"lr 2e-5 답변"`, `"lr 5e-6 답변"`이 차례로 붙고, 값은 빈 문자열이 아닌 문자열입니다.
- `small_lr_config`는 학습률, 폴더, run 이름만 `sft_config`와 다르고, `small_lr_output.global_step`이 34입니다.
- `trainer`, `first_model`, `small_lr_model`, `small_lr_trainer` 이름이 남아 있지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 첫 학습과 학습률 비교 절과 같은 순서입니다. 첫 학습 모델의 답을 열로 모으고 지운 뒤, 설정 세 줄만 바꿔 다시 학습하고, 같은 질문의 답을 열로 더합니다.

세부구현:
1. 테이블[새 열 이름] = answer_check_questions(모델)로 열을 더합니다.
2. 정리는 del, gc.collect, torch.cuda.empty_cache 순서입니다.
3. 4번 SFTConfig 셀의 값을 그대로 옮기고 세 줄만 바꿉니다. 바꾼 줄에 '바꿈' 주석을 달면 비교하기 쉽습니다.
4. lr 5e-6 답 열은 (1)과 같은 함수에 모델만 small_lr_model로 바꿉니다.
```

</details>

In [ ]:
# [작성] (1) 같은 질문 묶음을 lr 2e-5 모델에 묻고, (2) 모델과 학습기를 지우세요.

# (1) 1번과 같은 질문 묶음을 lr 2e-5 모델에 묻고, Base 답이 있는 테이블에 열로 더합니다
check_table["lr 2e-5 답변"] = ...
display(check_table)

# (2) 학습기(AdamW 갱신 상태 포함)와 lr 2e-5 모델을 지워 GPU 메모리를 비웁니다. 저장하지 않고, 답은 테이블에 남깁니다
...

In [ ]:
# [자가채점] lr 2e-5 답변 열과 지운 이름을 확인합니다
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "Base 답변", "lr 2e-5 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. 1번 테이블에 \"lr 2e-5 답변\" 열 하나를 더하세요."
for answer in check_table["lr 2e-5 답변"]:
    assert isinstance(answer, str) and answer.strip() != "", "lr 2e-5 답변이 비어 있습니다. answer_check_questions(first_model)의 반환값을 담으세요."

assert "trainer" not in globals() and "first_model" not in globals(), "답을 모은 뒤 trainer, first_model을 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")

In [ ]:
# [작성] (3) 학습률만 바꾼 설정으로 (4) 한 번 더 학습하세요.

# (3) sft_config에서 학습률, 폴더, run 이름만 바꿉니다
small_lr_config = SFTConfig(
    ...
)

# (4) 학습 전 상태의 모델을 새로 불러 같은 데이터로 학습합니다
small_lr_model = ...
small_lr_trainer = ...
small_lr_output = ...
display(small_lr_output)

In [ ]:
# [자가채점] 바꾼 설정과 끝까지 돈 학습을 확인합니다
assert isinstance(small_lr_config, SFTConfig), "small_lr_config는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert abs(small_lr_config.learning_rate - 5e-6) < 1e-12, f"small_lr_config의 learning_rate가 {small_lr_config.learning_rate}입니다. 5e-6를 넣으세요."
assert small_lr_config.run_name == "lr_5e-6" and small_lr_config.output_dir.endswith("commerce_lr5e-6"), "small_lr_config의 run_name은 \"lr_5e-6\", output_dir는 str(output_dir / \"commerce_lr5e-6\")입니다."
assert small_lr_config.num_train_epochs == sft_config.num_train_epochs and small_lr_config.max_steps == sft_config.max_steps and small_lr_config.per_device_train_batch_size == sft_config.per_device_train_batch_size, "학습률, 폴더, run 이름 말고는 sft_config와 같은 값이어야 비교가 됩니다(num_train_epochs, per_device_train_batch_size)."
assert small_lr_config.max_length == sft_config.max_length and small_lr_config.eval_steps == sft_config.eval_steps and small_lr_config.logging_steps == sft_config.logging_steps, "max_length, eval_steps, logging_steps도 sft_config와 같게 넣으세요."
assert small_lr_config.eval_strategy == "steps", "eval_strategy=\"steps\"도 sft_config와 같게 넣으세요. 넣지 않으면 검증 손실을 계산하지 않습니다."
assert small_lr_config.eval_on_start is True and "mlflow" in small_lr_config.report_to and small_lr_config.gradient_checkpointing is False, "eval_on_start=True, report_to=[\"mlflow\"], gradient_checkpointing=False도 sft_config와 같게 넣으세요."
assert small_lr_config.save_strategy == "no" and small_lr_config.seed == 0, "save_strategy=\"no\", seed=0도 sft_config와 같게 넣으세요."
assert next(small_lr_model.parameters()).dtype == torch.float32, "small_lr_model은 MODEL_NAME에서 dtype=torch.float32로 새로 불러옵니다."
assert small_lr_trainer.args is small_lr_config, "small_lr_trainer에 small_lr_config를 args로 넘기세요."
assert small_lr_output.global_step == 34, f"global_step이 {small_lr_output.global_step}입니다. small_lr_trainer.train()이 끝까지 돌았는지 확인하세요."
print("✅ 통과!")

In [ ]:
# [작성] (5) 같은 질문의 lr 5e-6 답을 더한 뒤, (6) 모델과 학습기를 지우세요.

# (5) 같은 질문 묶음을 lr 5e-6 모델에 묻고, Base, lr 2e-5 답이 있는 테이블에 열로 더합니다
check_table["lr 5e-6 답변"] = ...
display(check_table)

# (6) lr 5e-6 모델도 비교에만 썼으므로 저장하지 않고, 모델과 학습기를 지워 GPU 메모리를 비웁니다
...

In [ ]:
# [자가채점] lr 5e-6 답변 열과 지운 이름을 확인합니다
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "Base 답변", "lr 2e-5 답변", "lr 5e-6 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. \"lr 2e-5 답변\" 열 뒤에 \"lr 5e-6 답변\" 열 하나를 더하세요."
for answer in check_table["lr 5e-6 답변"]:
    assert isinstance(answer, str) and answer.strip() != "", "lr 5e-6 답변이 비어 있습니다. answer_check_questions(small_lr_model)의 반환값을 담으세요."
assert "small_lr_model" not in globals() and "small_lr_trainer" not in globals(), "답을 모은 뒤 small_lr_model, small_lr_trainer를 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")

## 7. 에폭 늘리기와 에폭별 체크포인트

**배경**: 교안 01의 에폭 늘리기와 같습니다. lr 5e-6로 4에폭을 끝까지 학습하며 에폭마다 체크포인트를 저장하고, 검증 손실은 기록만 합니다.

**요구사항**
- (1) `sft_config`와 같은 배치, `seed`, `gradient_checkpointing`, `max_length`에서 아래 값을 넣은 **`epoch_config`** 를 만듭니다. `max_steps`, `eval_steps`, `save_strategy="no"`는 넣지 않습니다.

```text
output_dir                   str(output_dir / "commerce_epochs")
learning_rate                5e-6
num_train_epochs             4
eval_strategy                "epoch"
save_strategy                "epoch"
save_only_model              True
logging_steps                10
eval_on_start                True
report_to                    ["mlflow"]
run_name                     "epochs4_lr_5e-6"
```

- (2) 학습 전 모델을 `MODEL_NAME`에서 `dtype=torch.float32`로 새로 불러 `device`에 올린 **`epoch_model`** 을 만듭니다(멈춤 토큰은 바꾸지 않습니다). `SFTTrainer`에 `epoch_model`, `epoch_config`, `train_dataset`, `eval_dataset=val_dataset`, `processing_class=tokenizer`를 넘긴 학습기 **`epoch_trainer`** 를 만들어 학습한 반환값을 **`epoch_output`** 에 담습니다.
- (3) `epoch_trainer.state.log_history`에서 검증 손실이 있는 줄만 골라 `step`, `epoch`, `eval_loss` 열(이 순서)만 둔 테이블 **`epoch_eval_log`** 를 만듭니다.
- (4) 한 에폭의 갱신 횟수 `steps_per_epoch`를 `epoch_config`의 배치 크기로 다시 계산하고, 1에폭부터 4에폭까지 체크포인트 폴더 `output_dir / "commerce_epochs" / f"checkpoint-{steps_per_epoch * epoch}"`의 에폭과 폴더 이름을 담은 행 리스트 **`checkpoint_rows`**(키 `"에폭"`, `"체크포인트"`, 폴더가 있는지 `checkpoint_dir.exists()`로 확인한 `"폴더 있음"`)와 테이블 **`checkpoint_table`** 을 만듭니다.
- (5) `epoch_model`, `epoch_trainer`를 지운 뒤 GPU 메모리를 비웁니다.

**확인 기준**
- `epoch_config`의 값이 위와 같고, `epoch_output.global_step`이 136(한 에폭 34번 * 4에폭)입니다.
- `epoch_eval_log`의 `step`이 [0, 34, 68, 102, 136](학습 전과 에폭 끝마다 한 행)이고, `checkpoint_table`의 `"체크포인트"`는 `checkpoint-` 뒤에 갱신 횟수 34, 68, 102, 136을 붙인 이름이며 `"폴더 있음"`은 모두 참입니다.
- 폴더마다 `config.json`이 있고 그 `"dtype"`이 `"float32"`입니다(float32로 불러 학습한 모델). `generation_config.json`의 `eos_token_id`에 두 토큰 id가 모두 들어 있습니다. `epoch_model`, `epoch_trainer` 이름이 남아 있지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01 에폭 늘리기 절의 학습, 에폭별 손실, 체크포인트 폴더 셀과 같은 순서입니다. 검증과 저장 간격을 에폭으로 맞춰 끝까지 학습하고, 폴더 이름의 갱신 횟수를 에폭 하나의 갱신 횟수로 계산합니다.

세부구현:
1. 위 값과 함께 per_device_train_batch_size, seed, gradient_checkpointing, max_length는 sft_config와 같게 넣습니다.
2. 멈춤 토큰은 바꾸지 않아도 됩니다. 학습기가 학습(train)을 시작할 때 토크나이저의 끝 토큰을 생성 설정에 더하고, 체크포인트에도 그대로 저장됩니다.
3. log_history로 만든 DataFrame에 dropna(subset=["eval_loss"])를 쓰고 세 열을 고릅니다.
4. del, gc.collect, torch.cuda.empty_cache 순서로 정리합니다.
```

</details>

In [ ]:
# [작성] (1) 에폭 설정으로 (2) 학습하고, (3) 에폭별 검증 손실과 (4) 체크포인트 폴더를 확인한 뒤 (5) 정리하세요.

# (1) 학습률, 에폭, 검증과 저장 간격을 바꿉니다. 중간에 멈추지 않고 끝까지 학습합니다
epoch_config = SFTConfig(
    ...
)

# (2) 학습 전 모델을 새로 불러옵니다
epoch_model = ...

# 같은 데이터로 학습기를 만들어 학습합니다
epoch_trainer = ...
epoch_output = ...
display(epoch_output)

# (3) 에폭별 검증 손실(맨 위는 학습 전). 학습 기록이고, 체크포인트를 고르는 기준이 아닙니다
epoch_log_table = pd.DataFrame(epoch_trainer.state.log_history)
epoch_eval_log = ...
display(epoch_eval_log.round(4))

# (4) 에폭 하나의 갱신 횟수 = 올림(학습 예시 수 / 배치)
steps_per_epoch = ...

# 1에폭부터 마지막 에폭까지 체크포인트 폴더 이름(폴더가 있는지도 확인합니다)
checkpoint_rows = []
for epoch in range(1, epoch_config.num_train_epochs + 1):
    checkpoint_dir = output_dir / "commerce_epochs" / f"checkpoint-{steps_per_epoch * epoch}"
    checkpoint_rows.append(...)

checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)

# (5) 에폭 학습의 모델과 학습기를 지워 체크포인트를 불러올 자리를 만듭니다
...

In [ ]:
# [자가채점] 에폭 설정, 끝까지 돈 학습, 에폭별 검증 손실, 체크포인트 폴더와 생성 설정, 정리를 확인합니다
assert isinstance(epoch_config, SFTConfig), "epoch_config는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert abs(epoch_config.learning_rate - 5e-6) < 1e-12, f"epoch_config의 learning_rate가 {epoch_config.learning_rate}입니다. 6번에서 비교한 5e-6를 넣으세요."
assert epoch_config.num_train_epochs == 4 and epoch_config.max_steps == -1, "num_train_epochs=4를 넣고 max_steps는 넣지 마세요(max_steps를 주면 에폭 수보다 앞섭니다)."
assert epoch_config.eval_strategy == "epoch" and epoch_config.save_strategy == "epoch", "eval_strategy와 save_strategy를 둘 다 \"epoch\"로 넣으세요."
assert epoch_config.save_only_model is True, "save_only_model=True를 넣으세요(가중치만 저장합니다)."
assert epoch_config.per_device_train_batch_size == 4 and epoch_config.max_length == 256 and epoch_config.seed == 0 and epoch_config.gradient_checkpointing is False, "per_device_train_batch_size, max_length, seed, gradient_checkpointing은 sft_config와 같게 넣으세요."
assert epoch_config.logging_steps == 10 and epoch_config.eval_on_start is True and "mlflow" in epoch_config.report_to, "logging_steps=10, eval_on_start=True, report_to=[\"mlflow\"]를 넣으세요."
assert epoch_config.run_name == "epochs4_lr_5e-6" and epoch_config.output_dir.endswith("commerce_epochs"), "run_name은 \"epochs4_lr_5e-6\", output_dir는 str(output_dir / \"commerce_epochs\")입니다. 과제 02가 이 이름으로 찾습니다."
assert epoch_output.global_step == 136, f"global_step이 {epoch_output.global_step}입니다(기대값 136). epoch_trainer.train()이 4에폭 끝까지 돌았는지 확인하세요."

assert list(epoch_eval_log.columns) == ["step", "epoch", "eval_loss"], f"epoch_eval_log의 열이 {list(epoch_eval_log.columns)}입니다. step, epoch, eval_loss 세 열을 이 순서로 고르세요."
assert list(epoch_eval_log["step"]) == [0, 34, 68, 102, 136], f"epoch_eval_log의 step이 {list(epoch_eval_log['step'])}입니다. 검증 손실이 있는 줄만 고르세요(eval_on_start=True면 step 0이 있습니다)."

# 에폭마다 저장한 체크포인트 폴더와 그 생성 설정
expected_eos = [END_OF_TURN_ID, tokenizer.convert_tokens_to_ids("<|endoftext|>")]
assert steps_per_epoch == 34, f"steps_per_epoch가 {steps_per_epoch}입니다(기대값 34). len(train_dataset)을 배치 크기로 나눠 math.ceil로 올리세요."
assert isinstance(checkpoint_table, pd.DataFrame) and list(checkpoint_table.columns) == ["에폭", "체크포인트", "폴더 있음"], "checkpoint_table은 키 \"에폭\", \"체크포인트\", \"폴더 있음\"인 행 리스트로 만든 테이블입니다."
assert checkpoint_table["폴더 있음"].all(), f"폴더 있음이 False인 에폭이 있습니다: {list(checkpoint_table['폴더 있음'])}. \"폴더 있음\"에는 checkpoint_dir.exists()를 담고, save_strategy=\"epoch\"로 학습했는지 확인하세요."
expected_names = []
for step in [34, 68, 102, 136]:
    expected_names.append(f"checkpoint-{step}")
assert list(checkpoint_table["체크포인트"]) == expected_names, f"체크포인트 열이 {list(checkpoint_table['체크포인트'])}입니다. 폴더 이름은 checkpoint-<한 에폭의 갱신 횟수 * 에폭>입니다."
for name in expected_names:
    checkpoint_dir = output_dir / "commerce_epochs" / name
    assert (checkpoint_dir / "config.json").exists(), f"{name} 폴더에 config.json이 없습니다. save_strategy=\"epoch\"를 넣었는지 확인하세요."
    checkpoint_config = json.loads((checkpoint_dir / "config.json").read_text(encoding="utf-8"))
    assert checkpoint_config.get("dtype") == "float32", f"{name}의 config.json dtype이 {checkpoint_config.get('dtype')}입니다. epoch_model을 MODEL_NAME에서 dtype=torch.float32로 불러 학습하세요."
    checkpoint_generation = json.loads((checkpoint_dir / "generation_config.json").read_text(encoding="utf-8"))
    for token_id in expected_eos:
        assert token_id in checkpoint_generation["eos_token_id"], f"{name}의 eos_token_id가 {checkpoint_generation['eos_token_id']}입니다. 학습기(SFTTrainer)로 학습하고 save_strategy=\"epoch\"로 저장했는지 확인하세요."
assert "epoch_model" not in globals() and "epoch_trainer" not in globals(), "확인이 끝나면 epoch_model, epoch_trainer를 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")

**에폭별 체크포인트의 답**

**배경**: 교안 01의 에폭별 체크포인트의 답과 같습니다. 1번과 같은 질문 묶음을 체크포인트마다 물어 같은 테이블에 열로 더합니다. 평가셋 채점은 과제 02에서 합니다.

**요구사항**
- (1) `checkpoint_rows`의 행마다 `output_dir / "commerce_epochs" / 행["체크포인트"]`를 `dtype=torch.float32`로 불러 `device`에 올린 **`checkpoint_model`** 을 만듭니다.
- (2) `answer_check_questions(checkpoint_model)`로 만든 답을 `check_table`의 새 열 `f"{행['에폭']}에폭 답변"`에 담습니다.
- (3) 체크포인트 하나를 다 쓰면 `checkpoint_model`을 지우고 GPU 메모리를 비운 뒤 다음 체크포인트를 불러옵니다.

**확인 기준**
- `check_table`에 `"1에폭 답변"`부터 `"4에폭 답변"`까지 열이 있고 값은 빈 문자열이 아닌 문자열입니다. `checkpoint_model` 이름이 남아 있지 않습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 01의 에폭별 체크포인트의 답 셀과 같습니다. for 문 안에서 불러오기, 답 모으기, 지우기를 차례로 합니다.

세부구현:
1. from_pretrained에 체크포인트 폴더 경로와 float32 자료형을 넘긴 뒤 device로 옮깁니다.
2. 열 이름은 f-string으로 에폭 번호를 넣어 만들고, 값은 answer_check_questions의 반환값입니다.
3. del, gc.collect, torch.cuda.empty_cache 순서로 정리합니다.
```

</details>

In [ ]:
# [작성] (1) 체크포인트를 하나씩 불러와 (2) 질문 묶음의 답을 열로 더하고 (3) 지우세요.
for checkpoint_row in checkpoint_rows:
    # (1) 에폭 체크포인트를 학습 전 모델과 같은 방법(float32)으로 불러옵니다
    checkpoint_model = ...

    # (2) 같은 질문 묶음의 답을 에폭 열로 더합니다
    check_table[f"{checkpoint_row['에폭']}에폭 답변"] = ...

    # (3) 다 쓴 체크포인트를 지워 다음 체크포인트를 불러올 자리를 만듭니다
    ...

# 종류, 질문, 정답 답변, Base 답 옆에 에폭별 답 열만 골라 봅니다(6번 두 학습률 모델의 답 열은 뺍니다)
epoch_columns = ["종류", "질문", "정답 답변", "Base 답변"]
for checkpoint_row in checkpoint_rows:
    epoch_columns.append(f"{checkpoint_row['에폭']}에폭 답변")
display(check_table[epoch_columns])

In [ ]:
# [자가채점] 에폭별 답변 열과 지운 모델을 확인합니다
for epoch in range(1, 4 + 1):
    column = f"{epoch}에폭 답변"
    assert column in check_table.columns, f"check_table에 \"{column}\" 열이 없습니다. 체크포인트마다 f\"{{행['에폭']}}에폭 답변\" 열을 더하세요."
    for answer in check_table[column]:
        assert isinstance(answer, str) and answer.strip() != "", f"\"{column}\"에 빈 답이 있습니다. answer_check_questions(checkpoint_model)의 반환값을 담으세요."
assert "checkpoint_model" not in globals(), "다 쓴 checkpoint_model을 del로 지워 GPU 메모리를 비우세요."
print("✅ 통과!")